<a href="https://colab.research.google.com/github/Eman-Adly/Eman-Adly/blob/main/Finalbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Embedding, Flatten, Dense, Input, Add
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder

In [10]:
df = pd.read_csv('/content/updated_book_recommendation_data.csv')

In [11]:
df

,student_id,book_id,author,title,rating,department,genre,year_of_study,published_year,num_pages,reading_frequency
0,8538,1544,Kyle Alvarado,Maintain interest manage evidence,2,Mathematics,Fantasy,4,2000,165,High
1,6131,2685,Chloe Medina,Reality free,2,Medicine,Science Fiction,2,1963,589,Low
2,1734,7678,William Edwards,Prepare management kitchen cut,5,Engineering,Educational,1,2007,751,High
3,2108,9053,Todd Steele,Modern moment provide receive image,2,Business,Mystery,4,1989,399,Low
4,3772,1641,Lisa Joseph,Traditional establish couple usually,5,History,Science Fiction,3,2018,528,Medium
...,...,...,...,...,...,...,...,...,...,...,...
1995,3844,8390,James Cherry,Hospital then,2,Physics,Non-Fiction,2,1967,912,Medium
1996,9831,3759,Anthony Haley,Laugh stuff there home sense,5,History,Thriller,1,1997,789,Medium
1997,1259,1047,Maria Adams,Step let entire reality course,3,Computer Science,Fantasy,2,1970,614,Low
1998,8720,8976,Susan Shepherd,Local become,2,Philosophy,Historical,2,1974,159,Low


In [15]:
df.isnull().sum()

,0
student_id,0
book_id,0
author,0
title,0
rating,0
department,0
genre,0
year_of_study,0
published_year,0
num_pages,0


In [16]:
df.duplicated().sum()

0

In [17]:
df.dropna(inplace=True)

In [6]:
# Create interaction matrix
interaction_matrix = df.pivot(index="student_id", columns="title", values="rating").fillna(0)
interaction_sparse = csr_matrix(interaction_matrix)
student_similarity = cosine_similarity(interaction_sparse)
student_sim_df = pd.DataFrame(student_similarity, index=interaction_matrix.index, columns=interaction_matrix.index)

# Encode categorical data for neural network
student_encoder = LabelEncoder()
book_encoder = LabelEncoder()
df['student_id_encoded'] = student_encoder.fit_transform(df['student_id'])
df['book_id_encoded'] = book_encoder.fit_transform(df['title'])

# Neural Network Model for Deep Learning-based CF
def build_deep_cf_model(n_students, n_books, embedding_size=50):
    student_input = Input(shape=(1,))
    book_input = Input(shape=(1,))

    student_embedding = Embedding(n_students, embedding_size)(student_input)
    book_embedding = Embedding(n_books, embedding_size)(book_input)

    student_vec = Flatten()(student_embedding)
    book_vec = Flatten()(book_embedding)

    merged = Add()([student_vec, book_vec])
    merged = Dense(128, activation='relu')(merged)
    output = Dense(1, activation='linear')(merged)

    model = Model(inputs=[student_input, book_input], outputs=output)
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse')
    return model

n_students = df['student_id_encoded'].nunique()
n_books = df['book_id_encoded'].nunique()
model = build_deep_cf_model(n_students, n_books)

# Hybrid Recommendation System    new student
def recommend_books_hybrid(student_id, student_department, student_year, top_n=5):
    if student_id not in interaction_matrix.index:
        print("New Student")
        top_books = df[df["department"] == student_department].sort_values(by="rating", ascending=False).head(top_n)
        return top_books[["author", "title", "genre", "published_year", "num_pages", "rating"]]

    # Collaborative Filtering: Similar Student-based recommendation   old student
    similar_students = student_sim_df[student_id].sort_values(ascending=False)[1:6]
    similar_students_books = interaction_matrix.loc[similar_students.index]
    book_recommendations = similar_students_books.mean(axis=0)

    # Exclude books already read
    books_read_by_student = interaction_matrix.loc[student_id]
    unread_books = book_recommendations[books_read_by_student == 0]

    # Deep Learning CF: Predict ratings
    student_encoded = student_encoder.transform([student_id])[0]
    book_candidates = unread_books.index
    book_encoded = book_encoder.transform(book_candidates)
    predicted_ratings = model.predict([np.array([student_encoded]*len(book_encoded)), np.array(book_encoded)])

    # Rank books based on predicted ratings
    book_scores = dict(zip(book_candidates, predicted_ratings.flatten()))
    top_books_cf = sorted(book_scores, key=book_scores.get, reverse=True)[:top_n]

    # Hybrid: Combine Content Features
    content_books = df[df['title'].isin(top_books_cf)].sort_values(by="rating", ascending=False)
    return content_books[["title","author", "genre", "published_year", "num_pages", "rating"]].drop_duplicates()

# Example usage
#print(recommend_books_hybrid(8538, "Mathematics", 4))


In [7]:
print(recommend_books_hybrid(8538, "Mathematics", 4))

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
                                             title             author  \
1447  Meet management education director president    Jennifer Oliver   
1000                 Center significant generation  Stephanie Johnson   
1425          Science stay production particularly      Donald Norris   
763                     Thus admit whom push never      Carl Gonzalez   
1516                                   Either foot     Justin Summers   

                genre  published_year  num_pages  rating  
1447          Mystery            1972        474       5  
1000  Science Fiction            1999        389       4  
1425         Thriller            1969        794       2  
763         Biography            1956        994       1  
1516         Thriller            1967        837       1  


In [8]:
print(recommend_books_hybrid(8, "Mathematics", 4))


New Student
                author                        title        genre  \
1991    Brittany Young                   Issue able    Biography   
348       William Cole   Development parent boy bar  Non-Fiction   
476      Brenda Howell         Writer suddenly lead      Fantasy   
1448  Nicholas Wilkins  Individual father pull step  Educational   
569       Donna Levine                  Water trade      Fantasy   

      published_year  num_pages  rating  
1991            1974        731       5  
348             1995        746       5  
476             1951        112       5  
1448            2007        604       5  
569             2017        292       5  


In [12]:
print(recommend_books_hybrid(6131, "Medicine	", 2))


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
                                     title             author  \
158                        Join win behind         Tammy Bell   
752                    Result ability book   Christopher Koch   
1000         Center significant generation  Stephanie Johnson   
1425  Science stay production particularly      Donald Norris   
942                       Might nature low       Angela Lewis   

                genre  published_year  num_pages  rating  
158           Mystery            2014        239       5  
752       Non-Fiction            1974        512       4  
1000  Science Fiction            1999        389       4  
1425         Thriller            1969        794       2  
942       Non-Fiction            2007        426       1  


In [14]:
print(recommend_books_hybrid(2, "Medicine", 4))


New Student
                    author                       title            genre  \
416         Zachary Wilson    Minute economy just case         Thriller   
293   Stephanie Williamson       Identify rule believe          Fantasy   
1349           Krista Todd            Section law door        Biography   
1812        Joanne Marquez           For instead which  Science Fiction   
455            Jerry Smith  Quality tonight degree tax      Educational   

      published_year  num_pages  rating  
416             2005        138       5  
293             2020        469       5  
1349            2023        890       5  
1812            2001        984       5  
455             1953        974       5  


## **API**

In [18]:
!pip install fastapi uvicorn pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 3.6 MB/s eta 0:00:00


In [19]:
!ngrok config add-authtoken 2uHIwaAOnVw5L7VhJrrCUdrUunc_3HuiMBikP5shbd2z1Ljfi

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [20]:
!pip install fastapi uvicorn pyngrok nest_asyncio

In [ ]:
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
import uvicorn
from pyngrok import ngrok
import nest_asyncio

app = FastAPI()

@app.get("/")
def home():
    return {"message": "API is running on Colab!"}

@app.get("/recommend")
def recommend_books():
    return {"books": ["Book 1", "Book 2", "Book 3"]}

nest_asyncio.apply()

port = 8000
public_url = ngrok.connect(port).public_url
print(f"🚀 Public URL: {public_url}")

uvicorn.run(app, host="0.0.0.0", port=port)

🚀 Public URL: https://74a2-34-139-98-114.ngrok-free.app


INFO:     Started server process [7676]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     149.154.161.248:0 - "GET / HTTP/1.1" 200 OK
INFO:     41.129.63.8:0 - "GET / HTTP/1.1" 200 OK
INFO:     41.129.63.8:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     45.244.46.255:0 - "GET / HTTP/1.1" 200 OK
